In [ ]:
# INSTALL & IMPORTS
# ================================
!pip install earthengine-api geemap geopandas

import ee
import geemap
import geopandas as gpd

# Authenticate
ee.Authenticate()
ee.Initialize(project="xyz")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ================================
# LOAD SHAPEFILE
# ================================
gdf = gpd.read_file('/content/drive/MyDrive/Indian_roads/Road.shp')
roads = geemap.geopandas_to_ee(gdf)

print("Road classes:", gdf['fclass'].unique())



In [ ]:
print(gdf['fclass'].unique())


['unclassified' 'tertiary' 'footway' 'residential' 'primary' 'trunk'
 'secondary' 'track' 'motorway_link' 'secondary_link' 'service' 'motorway'
 'primary_link' 'path' 'living_street' 'tertiary_link' 'pedestrian'
 'steps' 'trunk_link' 'bridleway' 'unknown' 'track_grade5' 'busway'
 'track_grade2' 'track_grade1' 'cycleway' 'track_grade3' 'track_grade4']


In [ ]:
# ================================
# PARAMETERS
# ================================
scale = 10
patch_pixels = 256
patch_size = scale * patch_pixels  # 2560 meters

threshold_main = 0.02
threshold_other = 0.001

start_date = '2025-03-01'
end_date = '2025-04-30'


In [ ]:
# ================================
# SENTINEL-2 PREP
# ================================
def maskS2clouds(image):
    qa = image.select('QA60')
    cloud = (1 << 10)
    cirrus = (1 << 11)
    mask = qa.bitwiseAnd(cloud).eq(0).And(
        qa.bitwiseAnd(cirrus).eq(0)
    )
    return image.updateMask(mask)

s2_rgb = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(roads)
    .filterDate(start_date, end_date)
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 5))
    .map(maskS2clouds)
    .median()
    .select(['B4', 'B3', 'B2'])
)

aoi = roads.geometry().bounds()


In [ ]:
# ================================
# SELECT ONLY REQUIRED CLASSES
# ================================
selected_classes = [
    'motorway',
    'trunk',
    'primary',
    'secondary',
    'tertiary',
    'unclassified'
]

# Keep only classes that actually exist in your data
available_classes = list(gdf['fclass'].unique())

road_classes = [cls for cls in selected_classes if cls in available_classes]

print("Processing classes:", road_classes)

In [ ]:
# ================================
# FILTER & RASTERIZE
# ================================
def rasterize(fc):
    return fc.reduceToImage([], ee.Reducer.constant(1)).unmask(0)

road_fc = {}
road_img = {}

for cls in road_classes:
    fc = roads.filter(ee.Filter.eq('fclass', cls))
    
    # Skip empty collections (extra safety)
    if fc.size().getInfo() == 0:
        print(f"Skipping empty class: {cls}")
        continue
    
    road_fc[cls] = fc
    road_img[cls] = rasterize(fc)

# All roads (only selected ones)
combined_fc = ee.FeatureCollection([])
for cls in road_fc:
    combined_fc = combined_fc.merge(road_fc[cls])

all_roads_img = rasterize(combined_fc)

In [ ]:
# ================================
# PATCH FUNCTION
# ================================
def make_patch(pt):
    return pt.buffer(patch_size / 2).bounds()



In [ ]:
# ================================
# CORE FUNCTION PER CATEGORY
# ================================
def process_category(cls):

    print(f"Processing {cls}...")

    main_fc = road_fc[cls]
    main_img = road_img[cls]

    # Other roads
    other_img = all_roads_img.subtract(main_img).gt(0)

    # Sample along roads (FAST!)
    points = main_fc.randomPoints(
        region=aoi,
        points=8000,   # oversample to ensure 500 valid
        seed=42
    )

    # Compute fractions
    def addFractions(f):
        geom = make_patch(f).geometry()

        main_frac = main_img.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=geom,
            scale=scale,
            maxPixels=1e8
        ).get('constant')

        other_frac = other_img.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=geom,
            scale=scale,
            maxPixels=1e8
        ).get('constant')

        return ee.Feature(geom).set({
            'main_frac': main_frac,
            'other_frac': other_frac
        })

    patches = (
        points.map(addFractions)
        .filter(
            ee.Filter.And(
                ee.Filter.gte('main_frac', threshold_main),
                ee.Filter.lte('other_frac', threshold_other)
            )
        )
        .limit(500)
    )


In [ ]:
# ================================
# CORE FUNCTION PER CATEGORY
# ================================
def process_category(cls):

    print(f"Processing {cls}...")

    main_fc = road_fc[cls]
    main_img = road_img[cls]

    # Other roads
    other_img = all_roads_img.subtract(main_img).gt(0)

    # Sample along roads (FAST!)
    points = main_fc.randomPoints(
        region=aoi,
        points=8000,   # oversample to ensure 500 valid
        seed=42
    )

    # Compute fractions
    def addFractions(f):
        geom = make_patch(f).geometry()

        main_frac = main_img.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=geom,
            scale=scale,
            maxPixels=1e8
        ).get('constant')

        other_frac = other_img.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=geom,
            scale=scale,
            maxPixels=1e8
        ).get('constant')

        return ee.Feature(geom).set({
            'main_frac': main_frac,
            'other_frac': other_frac
        })

    patches = (
        points.map(addFractions)
        .filter(
            ee.Filter.And(
                ee.Filter.gte('main_frac', threshold_main),
                ee.Filter.lte('other_frac', threshold_other)
            )
        )
        .limit(500)
    )

    # ================================
    # EXPORT (TFRecord - BEST)
    # ================================
    task = ee.batch.Export.image.toDrive(
        image=s2_rgb,
        description=f"{cls}_patches",
        folder=f"S2_{cls}",
        region=patches.geometry(),
        scale=10,
        fileFormat='TFRecord',
        formatOptions={
            'patchDimensions': [256, 256],
            'compressed': True
        },
        maxPixels=1e13
    )

    task.start()
    print(f"Export started for {cls}")


# ================================
# RUN ALL CATEGORIES
# ================================
for cls in road_classes:
    if cls not in road_fc:
        continue
    process_category(cls)